# 2 — Full-parameter DPO of OLMo 2 1B on the BeeS dataset

This notebook DPO post-trains `allenai/OLMo-2-0425-1B-SFT`; it does not perform another SFT
objective. Every model parameter is updated. The two GPUs train simultaneously with FSDP full
sharding, sized for the smaller 8 GB RTX 2080 SUPER.

Memory reductions are training-system techniques, not lossy model compression: FP32 master
parameters and updates with loss-scaled FP16 compute, full parameter/gradient sharding, activation
checkpointing/offloading, reference-log-probability precomputation, and paged **32-bit** AdamW
states. LoRA, PEFT, QLoRA, and 4/8-bit weight loading are explicitly rejected by the training script.
The conservative dynamic loss scale starts at 32, which was validated with actual 1,024-token pairs
on these GPUs. Every manifest records peak CUDA allocation/reservation for both ranks.

The original SFT model is never overwritten. A checkpoint is only recorded as approved after it
passes held-out preference checks and a mandatory multi-GPU general benchmark comparison.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys


def locate_bees() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd / "BeeS", cwd.parent, cwd.parent / "BeeS"):
        if (candidate / "ReadME.md").exists() and (candidate / "sampler.py").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from VPDPO, BeeS, or BeeS/notebooks")


PROJECT_ROOT = locate_bees()
WORKSPACE = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from olmo2_bees.common import (
    assert_two_turing_gpus,
    configure_workspace,
    package_versions,
    run_streaming,
    sha256_file,
    write_json,
)

configure_workspace(WORKSPACE)
print("BeeS repo:", PROJECT_ROOT)
print("Workspace:", WORKSPACE)


BeeS repo: /media/fezan/ASi/VPDPO/BeeS
Workspace: /media/fezan/ASi/VPDPO


In [2]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    run_streaming(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-olmo2.txt")],
        cwd=PROJECT_ROOT,
    )

gpus = assert_two_turing_gpus()
versions = package_versions(
    ["torch", "transformers", "datasets", "accelerate", "trl", "bitsandbytes", "lm_eval"]
)
print(json.dumps({"gpus": gpus, "packages": versions}, indent=2))


$ /media/fezan/ASi/VPDPO/.venv/bin/python -m pip install -r /media/fezan/ASi/VPDPO/BeeS/requirements-olmo2.txt

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
{
  "gpus": [
    {
      "index": 0,
      "name": "NVIDIA GeForce RTX 2080 Ti",
      "memory_gib": 10.74,
      "compute_capability": "7.5",
      "bf16_supported": false
    },
    {
      "index": 1,
      "name": "NVIDIA GeForce RTX 2080 SUPER",
      "memory_gib": 7.78,
      "compute_capability": "7.5",
      "bf16_supported": false
    }
  ],
  "packages": {
    "torch": "2.10.0",
    "transformers": "5.14.1",
    "datasets": "5.0.0",
    "accelerate": "1.14.0",
    "trl": "1.9.0",
    "bitsandbytes": "0.49.2",
    "lm_eval": "0.4.12"
  }
}


## Configuration and immutable inputs

The model and dataset revisions captured by notebook 1 are reused. The conservative `5e-7`,
`β=0.1`, two-epoch recipe follows the upstream BeeS defaults and established OLMo DPO scale.


In [3]:
MODEL_ID = "allenai/OLMo-2-0425-1B-SFT"
MAX_LENGTH = 1024
SEED = 42

ARTIFACTS = WORKSPACE / "artifacts" / "olmo2_bees"
DATA_ROOT = ARTIFACTS / "dataset_work"
SELECTED_DATASET = ARTIFACTS / "ultrafeedback_bees_olmo2_1b"
FINAL_RUN = ARTIFACTS / "olmo2_1b_dpo_full"
FINAL_MODEL = FINAL_RUN / "final"
PREFERENCE_EVAL = ARTIFACTS / "preference_eval"
BASELINE_LM_EVAL_ROOT = ARTIFACTS / "lm_eval_baseline"
CANDIDATE_LM_EVAL_ROOT = ARTIFACTS / "lm_eval_candidate"
QUALITY_REPORT = ARTIFACTS / "quality_gate.json"
APPROVED_POINTER = ARTIFACTS / "approved_model.json"

prepare_metadata = json.loads((DATA_ROOT / "prepare_metadata.json").read_text())
selection_metadata = json.loads((SELECTED_DATASET / "metadata.json").read_text())
MODEL_REVISION = prepare_metadata["model_revision"]
assert prepare_metadata["model_id"] == MODEL_ID
assert prepare_metadata["max_length"] == MAX_LENGTH
print(json.dumps({
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset": str(SELECTED_DATASET),
    "output": str(FINAL_MODEL),
}, indent=2))


{
  "model": "allenai/OLMo-2-0425-1B-SFT",
  "model_revision": "0d85a3d037876ce6ac7d4311d994400fc66ac27f",
  "dataset": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b",
  "output": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/olmo2_1b_dpo_full/final"
}


## Fail-closed dataset and hardware audit

The selected pairs must fit without truncation and must have positive external and implicit margins.
GPU 1 (8 GB) determines the per-device batch and sequence limits for both FSDP ranks.


In [4]:
from datasets import load_from_disk
from tqdm.auto import tqdm

dataset = load_from_disk(str(SELECTED_DATASET))
required = {"prompt", "chosen", "rejected", "external_margin", "implicit_margin", "bees_probability"}
assert required.issubset(dataset["train"].column_names)
assert len(dataset["train"]) == selection_metadata["selected_train_rows"]

for row in tqdm(dataset["train"], desc="Final pre-training audit"):
    assert row["max_pair_tokens"] <= MAX_LENGTH
    assert row["external_margin"] > 0.0
    assert row["implicit_margin"] > 0.0
    assert row["chosen"][0]["role"] == "assistant"
    assert row["rejected"][0]["role"] == "assistant"

minimum_gpu_gib = min(gpu["memory_gib"] for gpu in gpus)
assert minimum_gpu_gib >= 7.5
print(dataset)
print(f"Smaller GPU usable capacity: {minimum_gpu_gib:.2f} GiB")
print("Audit passed: no selected example requires truncation.")


Final pre-training audit:   0%|          | 0/6000 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'score_chosen', 'score_rejected', 'row_id', 'external_margin', 'prompt_tokens', 'chosen_tokens', 'rejected_tokens', 'max_pair_tokens', 'implicit_margin', 'external_preference_probability', 'implicit_preference_probability', 'bees_probability', 'bees_rank'],
        num_rows: 6000
    })
    test: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'score_chosen', 'score_rejected', 'row_id', 'external_margin', 'prompt_tokens', 'chosen_tokens', 'rejected_tokens', 'max_pair_tokens'],
        num_rows: 1891
    })
})
Smaller GPU usable capacity: 7.78 GiB
Audit passed: no selected example requires truncation.


## Train on both GPUs

The global batch is 16 preference pairs. Reference log-probabilities are precomputed while the model
still equals the pinned SFT checkpoint, then the same model becomes the trainable policy. This is
mathematically the normal fixed-reference DPO objective without retaining a second 1B model during
backpropagation.


In [5]:
if (FINAL_RUN / "training_manifest.json").is_file() and (FINAL_MODEL / "model.safetensors").is_file():
    print("Final model already exists; skipping training:", FINAL_MODEL)
else:
    train_command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--multi_gpu", "--num_processes", "2", "--gpu_ids", "0,1",
        "--mixed_precision", "fp16",
        "-m", "olmo2_bees.train_dpo",
        "--workspace", str(WORKSPACE),
        "--dataset-path", str(SELECTED_DATASET),
        "--train-split", "train",
        "--eval-split", "test",
        "--model-id", MODEL_ID,
        "--model-revision", MODEL_REVISION,
        "--output-dir", str(FINAL_RUN),
        "--run-name", "olmo2-1b-ultrafeedback-bees-dpo",
        "--max-length", str(MAX_LENGTH),
        "--epochs", "2",
        "--learning-rate", "5e-7",
        "--beta", "0.1",
        "--gradient-accumulation-steps", "8",
        "--logging-steps", "1",
        "--save-steps", "100",
        "--eval-steps", "100",
        "--num-proc", "4",
        "--seed", str(SEED),
        "--resume",
    ]
    run_streaming(train_command, cwd=PROJECT_ROOT)

manifest = json.loads((FINAL_RUN / "training_manifest.json").read_text())
assert manifest["full_parameter_training"] is True
assert manifest["parameters"]["trainable_parameters"] == manifest["parameters"]["total_parameters"]
assert manifest["peft_or_lora"] is False
assert manifest["weight_quantization"] is None
assert manifest["optimizer_is_paged"] is True
assert manifest["optimizer_state_bits"] == 32
assert manifest["optimizer"] == "bitsandbytes.optim.PagedAdamW32bit"
assert manifest["fp16_initial_loss_scale"] == 32.0
assert manifest["master_parameter_dtype"] == "float32"
assert manifest["compute_dtype"] == "float16"
assert manifest["saved_weight_dtypes"] == ["F32"]
assert manifest["weight_files_sha256"]
assert manifest["parallelism"] == "FSDP2_FULL_SHARD"
assert manifest["world_size"] == 2
for filename, expected_digest in tqdm(
    manifest["weight_files_sha256"].items(), desc="Verifying final FP32 weights"
):
    assert sha256_file(FINAL_MODEL / filename) == expected_digest

weight_manifest = json.dumps(manifest["weight_files_sha256"], sort_keys=True).encode("utf-8")
model_fingerprint = hashlib.sha256(weight_manifest).hexdigest()
BASELINE_LM_EVAL = BASELINE_LM_EVAL_ROOT / MODEL_REVISION[:16]
CANDIDATE_LM_EVAL = CANDIDATE_LM_EVAL_ROOT / model_fingerprint[:16]
print("Verified model fingerprint:", model_fingerprint)
print(json.dumps(manifest, indent=2))


$ /media/fezan/ASi/VPDPO/.venv/bin/python -m accelerate.commands.launch --multi_gpu --num_processes 2 --gpu_ids 0,1 --mixed_precision fp16 -m olmo2_bees.train_dpo --workspace /media/fezan/ASi/VPDPO --dataset-path /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b --train-split train --eval-split test --model-id allenai/OLMo-2-0425-1B-SFT --model-revision 0d85a3d037876ce6ac7d4311d994400fc66ac27f --output-dir /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/olmo2_1b_dpo_full --run-name olmo2-1b-ultrafeedback-bees-dpo --max-length 1024 --epochs 2 --learning-rate 5e-7 --beta 0.1 --gradient-accumulation-steps 8 --logging-steps 1 --save-steps 100 --eval-steps 100 --num-proc 4 --seed 42 --resume
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`

Verifying final FP32 weights:   0%|          | 0/1 [00:00<?, ?it/s]

Verified model fingerprint: 4a025b376176db74691750ae580bc9120478dc43d63d40eb3fc1746e1d448aca
{
  "activation_checkpointing": true,
  "activation_offloading": true,
  "compute_dtype": "float16",
  "dataset_path": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b",
  "final_model": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/olmo2_1b_dpo_full/final",
  "fp16_initial_loss_scale": 32.0,
  "full_parameter_training": true,
  "hyperparameters": {
    "beta": 0.1,
    "dataset_path": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b",
    "epochs": 2.0,
    "eval_split": "test",
    "eval_steps": 100,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-07,
    "logging_steps": 1,
    "max_length": 1024,
    "max_steps": -1,
    "model_id": "allenai/OLMo-2-0425-1B-SFT",
    "model_revision": "0d85a3d037876ce6ac7d4311d994400fc66ac27f",
    "num_proc": 4,
    "output_dir": "/media/fezan/ASi/VPDPO/artifacts/olmo2_bees/olmo2_1b_dpo_full",
   

## Held-out preference evaluation

Both GPUs compare the trained policy with the untouched pinned SFT reference on the official filtered
`test_prefs` split. The result supplies DPO reward accuracy, mean reward margin, and direct
length-normalized preference accuracy for both models.


In [6]:
preference_scores = PREFERENCE_EVAL / "implicit_scores.jsonl"
if preference_scores.is_file() and (PREFERENCE_EVAL / "score_manifest.json").is_file():
    print("Held-out preference scores already exist:", preference_scores)
else:
    eval_command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--multi_gpu", "--num_processes", "2", "--gpu_ids", "0,1",
        "--mixed_precision", "fp16",
        "-m", "olmo2_bees.score_implicit",
        "--workspace", str(WORKSPACE),
        "--dataset-path", str(SELECTED_DATASET),
        "--split", "test",
        "--reference-model", MODEL_ID,
        "--reference-revision", MODEL_REVISION,
        "--policy-model", str(FINAL_MODEL),
        "--output-dir", str(PREFERENCE_EVAL),
        "--max-length", str(MAX_LENGTH),
    ]
    run_streaming(eval_command, cwd=PROJECT_ROOT)

# Early gate before spending time on the full benchmark suite.
preference_gate_command = [
    sys.executable, "-m", "olmo2_bees.quality_gate",
    "--workspace", str(WORKSPACE),
    "--preference-scores", str(preference_scores),
    "--output", str(ARTIFACTS / "preference_gate.json"),
]
run_streaming(preference_gate_command, cwd=PROJECT_ROOT)


$ /media/fezan/ASi/VPDPO/.venv/bin/python -m accelerate.commands.launch --multi_gpu --num_processes 2 --gpu_ids 0,1 --mixed_precision fp16 -m olmo2_bees.score_implicit --workspace /media/fezan/ASi/VPDPO --dataset-path /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b --split test --reference-model allenai/OLMo-2-0425-1B-SFT --reference-revision 0d85a3d037876ce6ac7d4311d994400fc66ac27f --policy-model /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/olmo2_1b_dpo_full/final --output-dir /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/preference_eval --max-length 1024
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.

Loading weights: 100%|██████████| 179/179 [00:00<00:00, 327.71it/s]

Loading weights: 100%|██████████| 179/179 [00:00<00:00, 31

## General-capability regression suite

This runs the same deterministic `lm-eval` tasks against the pinned SFT baseline and the DPO
candidate, using both GPUs in data-parallel inference. It is mandatory because preference accuracy
alone cannot establish that general or instruction-following accuracy was maintained. Full evaluation
is slow.

The gate permits at most a 1 percentage-point drop on any task and at most a 0.2 point macro drop.
Change the task list only if you have a documented alternative acceptance suite.


In [8]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "lm_eval[hf,ifeval]==0.4.12",
])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 2.8 MB/s  0:00:0036m-:--:--
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  DEPRECATION: Building 'langdetect' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'langdetect'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=704d8e047b36c48ce6bfef97fc62dc5c20792d30dfea648f06d9caa81ec3e22d
  Stored in directory: /media/fezan/ASi/VPDPO/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [immutabledict]



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


0

In [9]:
BENCHMARK_TASKS = [
    "arc_challenge",
    "hellaswag",
    "winogrande",
    "gsm8k",
    "leaderboard_ifeval",
]

from olmo2_bees.quality_gate import task_metrics


def benchmark_complete(output_dir: Path) -> bool:
    try:
        return set(BENCHMARK_TASKS).issubset(task_metrics(output_dir))
    except (FileNotFoundError, RuntimeError, json.JSONDecodeError):
        return False


def run_lm_eval(model_args: str, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--multi_gpu", "--num_processes", "2", "--gpu_ids", "0,1",
        "--mixed_precision", "fp16",
        "-m", "lm_eval", "run",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", *BENCHMARK_TASKS,
        "--batch_size", "1",
        "--apply_chat_template",
        "--seed", str(SEED),
        "--output_path", str(output_dir),
    ]
    run_streaming(command, cwd=PROJECT_ROOT)


if not benchmark_complete(BASELINE_LM_EVAL):
    run_lm_eval(
        f"pretrained={MODEL_ID},revision={MODEL_REVISION},dtype=float16",
        BASELINE_LM_EVAL,
    )
else:
    print("Baseline lm-eval results already exist:", BASELINE_LM_EVAL)

if not benchmark_complete(CANDIDATE_LM_EVAL):
    run_lm_eval(f"pretrained={FINAL_MODEL},dtype=float16", CANDIDATE_LM_EVAL)
else:
    print("Candidate lm-eval results already exist:", CANDIDATE_LM_EVAL)


$ /media/fezan/ASi/VPDPO/.venv/bin/python -m accelerate.commands.launch --multi_gpu --num_processes 2 --gpu_ids 0,1 --mixed_precision fp16 -m lm_eval run --model hf --model_args pretrained=allenai/OLMo-2-0425-1B-SFT,revision=0d85a3d037876ce6ac7d4311d994400fc66ac27f,dtype=float16 --tasks arc_challenge hellaswag winogrande gsm8k leaderboard_ifeval --batch_size 1 --apply_chat_template --seed 42 --output_path /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/lm_eval_baseline/0d85a3d037876ce6
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-08-14:17:28:16 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-08-14:17:28:16 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-08-14:17:28:21 INFO   

## Final fail-closed approval gate

This cell stops with an error on regression. Only a passing run writes `approved_model.json`; the
pointer references the full, unquantized saved model and its evidence report without duplicating
several gigabytes of weights.


In [10]:
gate_command = [
    sys.executable, "-m", "olmo2_bees.quality_gate",
    "--workspace", str(WORKSPACE),
    "--preference-scores", str(preference_scores),
    "--output", str(QUALITY_REPORT),
]
gate_command.extend([
    "--baseline-lm-eval", str(BASELINE_LM_EVAL),
    "--candidate-lm-eval", str(CANDIDATE_LM_EVAL),
    "--max-task-drop", "0.01",
    "--max-macro-drop", "0.002",
])
for task in BENCHMARK_TASKS:
    gate_command.extend(["--required-benchmark-task", task])
run_streaming(gate_command, cwd=PROJECT_ROOT)

quality = json.loads(QUALITY_REPORT.read_text())
assert quality["passed"] is True
write_json(APPROVED_POINTER, {
    "approved": True,
    "model_path": str(FINAL_MODEL.resolve()),
    "base_model": MODEL_ID,
    "base_model_revision": MODEL_REVISION,
    "dataset_path": str(SELECTED_DATASET.resolve()),
    "training_manifest": str((FINAL_RUN / "training_manifest.json").resolve()),
    "model_fingerprint": model_fingerprint,
    "weight_files_sha256": manifest["weight_files_sha256"],
    "quality_report": str(QUALITY_REPORT.resolve()),
    "weight_quantization": None,
    "peft_or_lora": False,
})
print("Approved full-parameter model:", FINAL_MODEL)
print("Approval evidence:", APPROVED_POINTER)


$ /media/fezan/ASi/VPDPO/.venv/bin/python -m olmo2_bees.quality_gate --workspace /media/fezan/ASi/VPDPO --preference-scores /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/preference_eval/implicit_scores.jsonl --output /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/quality_gate.json --baseline-lm-eval /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/lm_eval_baseline/0d85a3d037876ce6 --candidate-lm-eval /media/fezan/ASi/VPDPO/artifacts/olmo2_bees/lm_eval_candidate/4a025b376176db74 --max-task-drop 0.01 --max-macro-drop 0.002 --required-benchmark-task arc_challenge --required-benchmark-task hellaswag --required-benchmark-task winogrande --required-benchmark-task gsm8k --required-benchmark-task leaderboard_ifeval
{
  "preference_metrics": {
    "rows": 1891.0,
    "dpo_reward_accuracy": 0.632469592808038,
    "implicit_margin_mean": 2.432956204094123,
    "implicit_margin_median": 1.2931976318359375,
    "policy_length_normalized_preference_accuracy": 0.5563194077207827,
    "reference_length_normaliz